# Feature Importance Example

This notebook is a learning example with comments. It shows several common ways to estimate feature importance:

- Decision Tree Gini importance
- Random Forest importance
- Permutation importance
- Logistic Regression coefficient importance

The notebook uses the breast cancer dataset from scikit-learn because it is already cleaned and works well for classification examples.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn import datasets
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

%matplotlib inline

## Load the dataset

We split the dataset into training and testing sets so we can train models on one part and evaluate them on another.

In [ ]:
dataset = datasets.load_breast_cancer()
X = pd.DataFrame(dataset.data, columns=dataset.feature_names)
y = pd.Series(dataset.target, name="diagnosis")

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y,
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
print("Number of features:", X.shape[1])
X.head()

In [ ]:
def show_top_importances(title, feature_names, scores, top_n=10):
    """Print the top features and show a bar chart."""
    importance_df = pd.DataFrame(
        {
            "feature": feature_names,
            "importance": scores,
        }
    ).sort_values("importance", ascending=False)

    print(f"\n{title}")
    print(importance_df.head(top_n).to_string(index=False))

    plt.figure(figsize=(10, 6))
    sns.barplot(
        data=importance_df.head(top_n),
        x="importance",
        y="feature",
        orient="h",
    )
    plt.title(title)
    plt.xlabel("Importance")
    plt.ylabel("Feature")
    plt.tight_layout()
    plt.show()

    return importance_df

## 1. Decision Tree Gini importance

A decision tree can rank features by how much they reduce Gini impurity while the tree is being built.

In [ ]:
decision_tree = DecisionTreeClassifier(
    criterion="gini",
    random_state=42,
    max_depth=4,
)
decision_tree.fit(X_train, y_train)

tree_predictions = decision_tree.predict(X_test)
tree_accuracy = accuracy_score(y_test, tree_predictions)

print("Decision Tree accuracy:", round(tree_accuracy, 3))
tree_importances = show_top_importances(
    "Decision Tree Gini Importances",
    X.columns,
    decision_tree.feature_importances_,
)
tree_importances.head(10)

## 2. Random Forest importance

A random forest averages feature importance across many trees, which is usually more stable than relying on a single tree.

In [ ]:
random_forest = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
)
random_forest.fit(X_train, y_train)

forest_predictions = random_forest.predict(X_test)
forest_accuracy = accuracy_score(y_test, forest_predictions)

print("Random Forest accuracy:", round(forest_accuracy, 3))
forest_importances = show_top_importances(
    "Random Forest Importances",
    X.columns,
    random_forest.feature_importances_,
)
forest_importances.head(10)

## 3. Permutation importance

Permutation importance shuffles one feature at a time and measures how much the model performance gets worse. If accuracy drops a lot, that feature was important.

In [ ]:
perm_result = permutation_importance(
    random_forest,
    X_test,
    y_test,
    n_repeats=10,
    random_state=42,
    scoring="accuracy",
)

perm_importances = show_top_importances(
    "Permutation Importances (Random Forest)",
    X.columns,
    perm_result.importances_mean,
)
perm_importances.head(10)

## 4. Logistic Regression coefficient importance

For logistic regression, coefficient size can act like a feature-importance measure, but only after scaling the input features so they are comparable.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

log_reg = LogisticRegression(max_iter=1000, random_state=42)
log_reg.fit(X_train_scaled, y_train)

log_predictions = log_reg.predict(X_test_scaled)
log_accuracy = accuracy_score(y_test, log_predictions)

# Absolute coefficient values are used so large positive and
# large negative effects are both treated as important.
logistic_importance = abs(log_reg.coef_[0])

print("Logistic Regression accuracy:", round(log_accuracy, 3))
log_importances = show_top_importances(
    "Logistic Regression Coefficient Importances",
    X.columns,
    logistic_importance,
)
log_importances.head(10)

## Interpretation

Different models can rank features differently because they define importance in different ways:

- tree-based importance comes from impurity reduction
- permutation importance comes from performance drop after shuffling
- logistic regression importance comes from coefficient size after scaling

That means it is normal for the top-ranked features to differ across methods.